In [0]:
from delta.tables import DeltaTable

In [0]:
passengers_day1 = [
(101,"Rahul Sharma","Hyderabad","Economy","India"),
(102,"Priya Reddy","Bangalore","Business","India"),
(103,"Amit Kumar","Mumbai","Economy","India"),
(104,"Sneha Patel","Delhi","Premium Economy","India"),
(105,"Farhan Ali","Chennai","Economy","India")
]
columns = [
 "passenger_id",
 "passenger_name",
 "city",
 "travel_class",
 "country"
]
df_day1 = spark.createDataFrame(
 passengers_day1,
 columns
)


In [0]:
passengers_day2 = [
(102,"Priya Reddy","Bangalore","First Class","India"),
(104,"Sneha Patel","Hyderabad","Premium Economy","India"),
(106,"Neha Singh","Pune","Economy","India"),
(107,"Arjun Verma","Kochi","Business","India")
]
df_day2 = spark.createDataFrame(
 passengers_day2,
 columns
)

In [0]:
delta_path = "dbfs:/FileStore/passengers_delta"

In [0]:
df_day1.write.format("delta").mode("overwrite").save("/tmp/passengers_delta")

In [0]:
spark.read.format("delta").load("/tmp/passengers_delta").count()

5

In [0]:
spark.read.format("delta").load("/tmp/passengers_delta").show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
from delta.tables import DeltaTable
DeltaTable.forPath(spark, "/tmp/passengers_delta").history().show(truncate=False)

+-------+-------------------+---------------+-------------------------------------------------------+---------+------------------------------------------------------------+----+------------------+------------------------------------+-----------------------+-----------+-----------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------+------------+------------------------------------------+
|version|timestamp          |userId         |userName                                               |operation|operationParameters                                         |job |notebook          |queryHistoryStatementId             |clusterId              |readVersion|isolationLevel   |isBlindAppend|operationMetrics                                                                                                                          |userMetadata|engineInfo                                |
+-----

In [0]:
DeltaTable.forPath(spark,"/tmp/passengers_delta").alias("t").merge(df_day2.alias("s"),"t.passenger_id=s.passenger_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.read.format("delta").load("/tmp/passengers_delta").filter("passenger_id=102").show()

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore| First Class|  India|
+------------+--------------+---------+------------+-------+



In [0]:
spark.read.format("delta").load("/tmp/passengers_delta").filter("passenger_id=106").show()

+------------+--------------+----+------------+-------+
|passenger_id|passenger_name|city|travel_class|country|
+------------+--------------+----+------------+-------+
|         106|    Neha Singh|Pune|     Economy|  India|
+------------+--------------+----+------------+-------+



In [0]:
spark.read.format("delta").option("versionAsOf",0).load("/tmp/passengers_delta").show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
spark.read.format("delta").load("/tmp/passengers_delta").show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
|         102|   Priya Reddy|Bangalore|    First Class|  India|
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
|         106|    Neha Singh|     Pune|        Economy|  India|
|         107|   Arjun Verma|    Kochi|       Business|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
print(spark.read.format("delta").option("versionAsOf",0).load("/tmp/passengers_delta").count(), spark.read.format("delta").load("/tmp/passengers_delta").count())

5 7


In [0]:
spark.read.format("delta").option("versionAsOf",0).load("/tmp/passengers_delta").filter("passenger_id=102").show()

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore|    Business|  India|
+------------+--------------+---------+------------+-------+



In [0]:
spark.read.format("delta").load("/tmp/passengers_delta").filter("passenger_id=102").show()

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore| First Class|  India|
+------------+--------------+---------+------------+-------+



In [0]:
spark.read.format("delta").option("versionAsOf",0).load("/tmp/passengers_delta").filter("passenger_id=104").show()

+------------+--------------+-----+---------------+-------+
|passenger_id|passenger_name| city|   travel_class|country|
+------------+--------------+-----+---------------+-------+
|         104|   Sneha Patel|Delhi|Premium Economy|  India|
+------------+--------------+-----+---------------+-------+



In [0]:
spark.read.format("delta").load("/tmp/passengers_delta").filter("passenger_id=104").show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
spark.sql("OPTIMIZE delta.`/tmp/passengers_delta`")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
spark.sql("OPTIMIZE delta.`/tmp/passengers_delta` ZORDER BY (city)")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
DeltaTable.forPath(spark,"/tmp/passengers_delta").delete("passenger_id=105")

DataFrame[num_affected_rows: bigint]

In [0]:
DeltaTable.forPath(spark,"/tmp/passengers_delta").history().show(truncate=False)

+-------+-------------------+---------------+-------------------------------------------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+-----------------------+-----------+-----------------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
spark.sql("VACUUM delta.`/tmp/passengers_delta` RETAIN 168 HOURS")

DataFrame[path: string]